In [1]:
from utils.spark_utils import initialize_spark, read_parquet
from utils.hadoop_config import configure_windows_hadoop
import os
from schemas.silver.dim_question_schema import build_dim_question_schema
from schemas.silver.dim_exam_schema import build_dim_exam_schema
from schemas.silver.dim_answer_schema import build_dim_answer_schema
from schemas.silver.dim_file_schema import build_dim_file_schema
from schemas.silver.fact_download_schema import build_fact_download_schema
from schemas.silver.fact_qa_schema import build_fact_qa_schema
from pyspark.sql import functions as F

In [2]:
configure_windows_hadoop()

In [3]:
spark = initialize_spark("test-read-silver")

In [4]:
SILVER_BY_EXAM_DIR = os.path.join(
    "..", "data", "silver", "topik_documents", "by_exam"
)

In [5]:
question_df = read_parquet(spark,SILVER_BY_EXAM_DIR,os.path.join("102nd","dim_question"),build_dim_question_schema())

In [6]:
exam_df = read_parquet(spark,SILVER_BY_EXAM_DIR,os.path.join("102nd","dim_exam"),build_dim_exam_schema())

In [7]:
answers_df = read_parquet(spark,SILVER_BY_EXAM_DIR,os.path.join("102nd","dim_answer"), build_dim_answer_schema())

In [8]:
files_df = read_parquet(spark,SILVER_BY_EXAM_DIR,os.path.join("102nd","dim_file"), build_dim_file_schema())

In [9]:
fact_download = read_parquet(spark, SILVER_BY_EXAM_DIR, os.path.join("102nd", "fact_download"), build_fact_download_schema())

In [10]:
fact_qa = read_parquet(spark, SILVER_BY_EXAM_DIR, os.path.join("102nd", "fact_qa"), build_fact_qa_schema())

In [11]:
question_df.show()

+--------------------+--------------------+-----------+--------------------+----------------------------------+------------------------------+----------------------------+-------------------------+----------------+----------+
|         question_id|             exam_id|page_number|       question_type|                 raw_question_text|     question_vocabulary_items|   question_grammar_patterns|                    topic|difficulty_level|confidence|
+--------------------+--------------------+-----------+--------------------+----------------------------------+------------------------------+----------------------------+-------------------------+----------------+----------+
|5f7bd324b70feec8b...|8cf82e7c-fa17-458...|          6|reading comprehen...|  지난달 문을 연 우표 박물관이 ...|    [박물관, 우표, 지시, 사진]|   [-는, -ㄴ 다., -고 있다.]|              우표 박물관|          medium|       0.6|
|7975a2c988727eda4...|8cf82e7c-fa17-458...|          7|reading comprehen...|          (가) 그래서 multicult...|    [복숭아, 과일, 식감, 장점]

In [33]:
exam_df.show()

+--------------------+--------------------+------------+-----------+-----------+------------+
|             exam_id|           exam_name|exam_edition|exam_number|topik_level|exam_section|
+--------------------+--------------------+------------+-----------+-----------+------------+
|8cf82e7c-fa17-458...|102nd TOPIK II - ...|       102nd|        102|   TOPIK II|     Reading|
|d1ffb1aa-c0c4-40c...|102nd TOPIK I - T...|       102nd|        102|    TOPIK I|     Reading|
+--------------------+--------------------+------------+-----------+-----------+------------+



In [36]:
question_df.select("exam_id","page_number","raw_question_text").join(exam_df,on="exam_id",how="inner").show(3,truncate=False)

+------------------------------------+-----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------+------------+-----------+-----------+------------+
|exam_id                             |page_number|raw_question_text                                                                                                                                     

In [39]:
exam_df.show(1)

+--------------------+--------------------+------------+-----------+-----------+------------+
|             exam_id|           exam_name|exam_edition|exam_number|topik_level|exam_section|
+--------------------+--------------------+------------+-----------+-----------+------------+
|8cf82e7c-fa17-458...|102nd TOPIK II - ...|       102nd|        102|   TOPIK II|     Reading|
+--------------------+--------------------+------------+-----------+-----------+------------+
only showing top 1 row


In [49]:
answers_df.join(question_df,on="question_id",how="left").select("page_number","raw_question_text","raw_answer_text").orderBy("page_number").show(3)

+-----------+--------------------------------+------------------+
|page_number|               raw_question_text|   raw_answer_text|
+-----------+--------------------------------+------------------+
|          5|민수 씨하고 약속이 있습니다. ...|        기다립니다|
|          5|       ※  [40~42] 다음을 읽고...|일요일은 많습니다.|
|          5| 선물을 삽니다. 친구에게（   ...|          가르침다|
+-----------+--------------------------------+------------------+
only showing top 3 rows
